In [2]:
import shutil, os

if os.path.exists("/content/ZeroStance"):
    shutil.rmtree("/content/ZeroStance")

!git clone https://github.com/chenyez/ZeroStance.git
%cd /content
!pip install -q transformers sentencepiece tweet-preprocessor wordninja tensorboard scikit-learn pandas tqdm

SRC = "/content/ZeroStance/src"
CONFIG_PATH = "/content/ZeroStance/config/config-roberta_large.txt"
DATA_ROOT = "/content/ZeroStance/data"

print("Repo cloned. Contents of data/:", os.listdir(DATA_ROOT))

Cloning into 'ZeroStance'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 229 (delta 4), reused 0 (delta 0), pack-reused 218 (from 2)
Receiving objects: 100% (229/229), 61.98 MiB | 18.74 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (212/212), done.
/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Repo cloned. Contents of data/: ['chatgpt_wtwt', 'openstance_vast', 'chatgpt_pstance', 'chatgpt_carto_var_0.99_seed0_subset_500', 'chatgpt', 'chatgpt_covid19', 'pstance', 'ibm30k', 'chatgpt_carto_var_0.99_seed0_subset_2000', 'chatgpt_carto_var_0.99_2types_35_seed0', '.DS_Store', 'chatgpt_carto_var_0.99_seed0_subset_1000', 'chatgpt_semeval2016', 'chatgpt_carto_bertweet_var_0.99_seed0', 'wtwt', 'covid19', 'vast', 'chatgpt_carto_var_0.99_2types_25_seed0', 'chatgpt_carto_var_0.99_1type_2_

In [3]:
import re, os

def read(path):
    with open(path, "r") as f:
        return f.read()

def write(path, content):
    with open(path, "w") as f:
        f.write(content)

# --- 1. AdamW import (both files that need it) ---
for path in [f"{SRC}/train_model_v2.py", f"{SRC}/utils/model_utils.py"]:
    c = read(path)
    c2 = c.replace("from transformers import AdamW", "from torch.optim import AdamW")
    write(path, c2)
    print(f"AdamW patch applied: {path}")

# --- 2. batch_encode_plus -> callable tokenizer ---
path = f"{SRC}/utils/data_helper.py"
c = read(path)
c2 = c.replace("tokenizer.batch_encode_plus(", "tokenizer(")

# --- 3. bertweet-large local path -> vinai/bertweet-base (quote/whitespace agnostic,
#         also strips local_files_only=True so it can actually download from the hub) ---
pattern = re.compile(
    r"""(Auto(?:Tokenizer|Config|Model))\.from_pretrained\(\s*['"]\.\./\.\./model_hub/bertweet-large['"](?:\s*,\s*local_files_only\s*=\s*True)?\s*\)"""
)
c2 = pattern.sub(r"\1.from_pretrained('vinai/bertweet-base')", c2)
write(path, c2)
print(f"data_helper.py patched.")

path = f"{SRC}/utils/modeling.py"
c = read(path)
c2 = pattern.sub(r"\1.from_pretrained('vinai/bertweet-base')", c)
write(path, c2)
print(f"modeling.py patched.")

# --- 4. np.Inf -> np.inf ---
path = f"{SRC}/pytorchtools.py"
c = read(path)
c2 = c.replace("np.Inf", "np.inf")
write(path, c2)
print("pytorchtools.py patched.")

# --- 5. max_tok_len 250 -> 128 (avoid OOM) ---
c = read(CONFIG_PATH)
c2 = c.replace("max_tok_len:250", "max_tok_len:128")
write(CONFIG_PATH, c2)
print("config patched.")

# --- Verify nothing problematic remains ---
import subprocess
check = subprocess.run(
    ["grep", "-rn", "model_hub\\|from transformers import AdamW\\|np.Inf",
     f"{SRC}/train_model_v2.py", f"{SRC}/utils/model_utils.py",
     f"{SRC}/utils/data_helper.py", f"{SRC}/utils/modeling.py", f"{SRC}/pytorchtools.py"],
    capture_output=True, text=True
)
print("\nRemaining matches (should be empty or only harmless commented/dead-code lines):")
print(check.stdout if check.stdout else "(none)")

AdamW patch applied: /content/ZeroStance/src/train_model_v2.py
AdamW patch applied: /content/ZeroStance/src/utils/model_utils.py
data_helper.py patched.
modeling.py patched.
pytorchtools.py patched.
config patched.

Remaining matches (should be empty or only harmless commented/dead-code lines):
/content/ZeroStance/src/utils/data_helper.py:43:#         tokenizer = BertTokenizer.from_pretrained("../../model_hub/covid-twitter-bert-v2")
/content/ZeroStance/src/utils/data_helper.py:44:        tokenizer = AutoTokenizer.from_pretrained("../../model_hub/twhin-bert-large")



In [4]:
import subprocess, sys, os

CKPT_PATH = f"{SRC}/RoBERTa_seed42.pt"

if os.path.exists(CKPT_PATH):
    print(f"Checkpoint already exists — skipping training.")
else:
    print("Training on COVID-19...")
    result = subprocess.run(
        [sys.executable, "train_model_v2.py",
         "-c", CONFIG_PATH,
         "-s", "42", "-d", "0.1",
         "-train", f"{DATA_ROOT}/",
         "-dev", f"{DATA_ROOT}/",
         "-test", f"{DATA_ROOT}/",
         "-dataset", "covid19",
         "-leave_one_out", "0",
         "-lr1", "1e-5", "-lr2", "1e-4",
         "-step", "1", "-es_step", "5"],
        cwd=SRC, capture_output=True, text=True
    )
    print(result.stdout[-2500:])
    if result.returncode != 0 and "IndexError: list index out of range" not in result.stderr:
        print("STDERR:", result.stderr[-2500:])
    print("Checkpoint exists now:", os.path.exists(CKPT_PATH))

Training on COVID-19...
0.7068       800

results_weighted: (0.711704803960926, 0.70885548041683, 0.7058026085305015, None)
result_against: [np.float64(0.6566037735849056), np.float64(0.7160493827160493), np.float64(0.6850393700787402)]
result_favor: [np.float64(0.67), np.float64(0.7642585551330798), np.float64(0.7140319715808171)]
result_neutral: [np.float64(0.8085106382978723), np.float64(0.6462585034013606), np.float64(0.718336483931947)]
result_overall: [0.711704803960926, 0.70885548041683, 0.7058026085305015]
result_combined: [['test', 'covid19', 284, 42, '0.1', np.float64(0.6566037735849056), np.float64(0.7160493827160493), np.float64(0.6850393700787402), np.float64(0.67), np.float64(0.7642585551330798), np.float64(0.7140319715808171), np.float64(0.8085106382978723), np.float64(0.6462585034013606), np.float64(0.718336483931947), 0.711704803960926, 0.70885548041683, 0.7058026085305015]]
./results_test_df.csv saved!
------------------------------------------------------------------

In [5]:
import sys, json, torch, torch.nn as nn, pandas as pd, os
sys.path.insert(0, SRC)

from sklearn.metrics import precision_recall_fscore_support, classification_report
import utils.data_helper as dh
import utils.preprocessing as pp
from utils import model_utils

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_NAMES_3 = ["Against", "Favor", "Neutral"]

def build_config(config_file):
    config = {}
    with open(config_file, "r") as f:
        for l in f.readlines():
            if ":" in l.strip():
                k, v = l.strip().split(":", 1)
                config[k] = v
    return config

def load_norm_dict():
    norm_dict = {}
    p1, p2 = f"{SRC}/noslang_data.json", f"{SRC}/emnlp_dict.txt"
    if os.path.exists(p1):
        with open(p1) as f: norm_dict.update(json.load(f))
    if os.path.exists(p2):
        with open(p2) as f:
            for line in f.readlines():
                if "\t" in line:
                    row = line.split("\t")
                    norm_dict[row[0]] = row[1].rstrip()
    return norm_dict

config = build_config(CONFIG_PATH)
norm_dict = load_norm_dict()

model, _ = model_utils.model_setup(3, config["model_select"], DEVICE, config, float(config.get("dropout", "0.1")))
state = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state, strict=True)
model.to(DEVICE)
model.eval()
print(f"Loaded checkpoint on {DEVICE}")

def evaluate_on_dataset(dataset_name):
    test_path = os.path.join(DATA_ROOT, dataset_name, "raw_test_all_onecol.csv")
    if not os.path.exists(test_path):
        print(f"SKIP {dataset_name}: {test_path} not found.")
        return None
    x_test, y_test, x_test_target = pp.clean_all(test_path, norm_dict, dataset_name, dataset_name)
    dummy = [x_test, y_test, x_test_target]
    loader, gt_label = dh.data_helper_bert(dummy, dummy, dummy, config["model_select"], config)
    testloader = loader[2]
    y_test_gt = gt_label[2].to(DEVICE)
    loss_function = nn.CrossEntropyLoss()
    with torch.no_grad():
        preds_test, _ = model_utils.model_preds(testloader, model, DEVICE, loss_function)
    preds_np = preds_test.cpu().numpy().argmax(axis=1)
    y_np = y_test_gt.cpu().numpy()
    present = sorted(set(y_np.tolist()) | set(preds_np.tolist()))
    names = [LABEL_NAMES_3[i] for i in present]
    print("=" * 60)
    print(f"Dataset: {dataset_name} (n={len(y_np)})")
    print(classification_report(y_np, preds_np, labels=present, target_names=names, digits=4))
    _, _, f1, _ = precision_recall_fscore_support(y_np, preds_np, labels=present, average="macro")
    print(f"Macro-F1: {f1:.4f}")
    return {"dataset": dataset_name, "n": len(y_np), "f1_macro": f1}

results = [{"dataset": "covid19 (in-domain)", "n": 800, "f1_macro": 0.7058}]
for ds in ["vast", "semeval2016", "wtwt"]:
    r = evaluate_on_dataset(ds)
    if r:
        results.append(r)

results_df = pd.DataFrame(results)
results_df.to_csv("/content/cross_dataset_results.csv", index=False)
print("\nFINAL TABLE")
print(results_df.to_string(index=False))

BERTweet is used as the stance classifier.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded checkpoint on cuda


/content/ZeroStance/src/utils/preprocessing.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  label = pd.DataFrame.replace(raw_label, ['AGAINST', 'FAVOR', 'NONE'], [0, 1, 2])


Loading data
Dataset: vast (n=1460)
              precision    recall  f1-score   support

     Against     0.3734    0.3551    0.3640       490
       Favor     0.4400    0.0732    0.1255       451
     Neutral     0.3863    0.6840    0.4937       519

    accuracy                         0.3849      1460
   macro avg     0.3999    0.3708    0.3277      1460
weighted avg     0.3986    0.3849    0.3364      1460

Macro-F1: 0.3277


/content/ZeroStance/src/utils/preprocessing.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  label = pd.DataFrame.replace(raw_label, ['AGAINST', 'FAVOR', 'NONE'], [0, 1, 2])


Loading data
Dataset: semeval2016 (n=1249)
              precision    recall  f1-score   support

     Against     0.5319    0.1049    0.1752       715
       Favor     0.5385    0.0230    0.0442       304
     Neutral     0.1936    0.9217    0.3200       230

    accuracy                         0.2354      1249
   macro avg     0.4213    0.3499    0.1798      1249
weighted avg     0.4712    0.2354    0.1700      1249

Macro-F1: 0.1798


/content/ZeroStance/src/utils/preprocessing.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  label = pd.DataFrame.replace(raw_label, ['AGAINST', 'FAVOR', 'NONE'], [0, 1, 2])


Loading data
Dataset: wtwt (n=3037)
              precision    recall  f1-score   support

     Against     0.2768    0.4704    0.3485       423
       Favor     0.0000    0.0000    0.0000       666
     Neutral     0.6437    0.7659    0.6995      1948

    accuracy                         0.5568      3037
   macro avg     0.3068    0.4121    0.3493      3037
weighted avg     0.4514    0.5568    0.4972      3037

Macro-F1: 0.3493

FINAL TABLE
            dataset    n  f1_macro
covid19 (in-domain)  800  0.705800
               vast 1460  0.327744
        semeval2016 1249  0.179799
               wtwt 3037  0.349332


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m